In [8]:
import os
import json
import shutil
from tqdm import tqdm

In [9]:
SOURCE_IMAGE_DIR = "./validation/image"
SOURCE_ANNO_DIR = "./validation/annos"

In [10]:
TARGET_ROOT = "./DeepFashion2_Top5_val"
TARGET_IMAGE_DIR = os.path.join(TARGET_ROOT, "images")
TARGET_ANNO_DIR = os.path.join(TARGET_ROOT, "annos")

In [11]:
os.makedirs(TARGET_IMAGE_DIR, exist_ok=True)
os.makedirs(TARGET_ANNO_DIR, exist_ok=True)

In [12]:
TOP5 = [1, 8, 7, 2, 9]

In [13]:
kept_images = 0
skipped_images = 0

json_files = os.listdir(SOURCE_ANNO_DIR)

for file in tqdm(json_files):
    anno_path = os.path.join(SOURCE_ANNO_DIR, file)
    
    with open(anno_path, "r") as f:
        data = json.load(f)
    
    new_data = {}
    new_data["source"] = data.get("source", "")
    new_data["pair_id"] = data.get("pair_id", "")
    
    item_index = 1
    valid_item_found = False
    
    for key in data:
        if key.startswith("item"):
            item = data[key]
            
            if item["category_id"] in TOP5:
                new_data[f"item{item_index}"] = item
                item_index += 1
                valid_item_found = True
    
    # If no valid items, skip image
    if not valid_item_found:
        skipped_images += 1
        continue
    
    # Save filtered annotation
    target_anno_path = os.path.join(TARGET_ANNO_DIR, file)
    with open(target_anno_path, "w") as f:
        json.dump(new_data, f)
    
    # Copy corresponding image
    img_name = file.replace(".json", ".jpg")
    source_img_path = os.path.join(SOURCE_IMAGE_DIR, img_name)
    target_img_path = os.path.join(TARGET_IMAGE_DIR, img_name)
    
    if os.path.exists(source_img_path):
        shutil.copy2(source_img_path, target_img_path)
        kept_images += 1
    else:
        print(f"Warning: Image not found for {file}")

print("\nFiltering Complete!")
print("Images kept:", kept_images)
print("Images skipped:", skipped_images)


Filtering Complete!
Images kept: 23741
Images skipped: 8412
